# Notebook 2: Creating Embeddings

In this notebook you will learn how to build embeddings from scratch:
sentence → document → topic.

In [ ]:
from pathlib import Path
import sys

sys.path.append("../")

# Check imports
from src.config import REPO

In [5]:
# Tear Down/ Reset the state of the processed posts to start fresh.
from src.config import OUTPUT

def remove_processed_posts():
    """Remove the processed posts to start fresh."""
    processed_path = OUTPUT / "processed_posts.json"
    if processed_path.exists():
        processed_path.unlink()

remove_processed_posts()

## Outline:
- What is a dense vector?
- What is an "embedding"
- How text is converted into a dense vector (embedding) using a pre-trained model.
- Explanation of "embedding space" and what is happening under the hood...
- Creating a simple embedding function using a pre-trained model from Hugging Face.
- Code to generate a word embedding, a sentence embedding and a document embedding.
    - Understanding the difference between these three types of embeddings.
    - What shape do you get for each type of embedding? Why? What does it mean?
    - Why it makes sense to use sentence embeddings as opppsed to word embeddings for our use case.
- Why do we "preprocess" text before passing it to the embedding model?
     - What does preprocessing do?
     - How do we "clean" text?
     - Why is it important?
     - What do we mean by "structured" document vs and unstructured text
     - What does the "structured" document look like in our case?


## 1. What is an embedding?

A **sparse** representation of "cat" in a vocabulary of 30,000 words is a one-hot vector
that is 0 everywhere and 1 at the index of "cat". It carries no information about *meaning*
— "cat" and "kitten" are as far apart as "cat" and "helicopter".

A **dense** embedding is a fixed-length list of floats — typically 384, 768, or 1024 values —
where every dimension carries a slice of meaning learned from a large corpus.
When we pass text to an embedding model such as 
`all-MiniLM-L6-v2` (the model we use throughout this workshop), it is
transformed into a numeric representation of the term (or terms) that carries associations
and patterns about how that term is used:

```
cat = [-0.0237, 0.0815, -0.0011, …, 0.0532]   # 384 floats
```

Two pieces of text whose meanings are similar end up close together in this 384-dimensional
space, even when they share no tokens. That is what makes semantic search work.

In [ ]:
from sentence_transformers import SentenceTransformer

# Same model used by src/preprocess.py and src/search.py (see src/config.py)
model = SentenceTransformer("all-MiniLM-L6-v2")

word     = "cat"
sentence = "The cat is sleeping on the keyboard."
document = (
    "The orange tabby cat napped on the warm keyboard for hours, occasionally "
    "swatting at the cursor before settling back into a contented purr. "
    "The owner gave up trying to type and made a cup of tea instead."
)

for label, text in [("word", word), ("sentence", sentence), ("document", document)]:
    embedding = model.encode(text, convert_to_numpy=True)
    print(f"{label:9s} shape={embedding.shape}  first 5 dims={embedding[:5]}")

### Why are all three the same shape?

`all-MiniLM-L6-v2` is a *sentence-transformer*. It is a type of neural network. Internally it tokenizes the input,
runs each token through 6 transformer layers to produce contextual token vectors,
and then **mean-pools** those token vectors into a single 384-dimensional vector that
represents the entire input.

Because of pooling, the output shape is always `(384,)` regardless of whether the
input is a single word or a paragraph. That's exactly what we want for search:
every document, no matter how long, becomes one comparable vector.

For our use case (short social-media posts) sentence-level embeddings are a good
match. For long-form documents you might prefer to chunk into sentences/paragraphs,
embed each chunk, and store them separately.

### Embedding space: similar meanings cluster together

Below we encode three short phrases and compute pairwise cosine similarity. Two of
them are about cats; one is about the stock market. The cat phrases share **no
tokens** but should still score high against each other.

In [ ]:
import numpy as np
import pandas as pd

phrases = ["cat purring", "kitten meowing", "stock market crash"]
embeddings = model.encode(phrases, convert_to_numpy=True)

# Normalize so the dot product equals cosine similarity
normed = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
similarity_matrix = normed @ normed.T

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=phrases,
    columns=phrases,
)

print("Cosine similarity between phrase embeddings:")
display(similarity_df.round(4))

### What is happening under the hood


beddEmbedding models translate a term (or "token") to a vector that is representative of is usage.
The model does not rely on (1,0) encoding for each word, it is based on a neural
network. When you pass text through the model, the final layer's pooled output
is a numeric vector.

They are called "embedding" models because they embed meaning into a vector
space — they map discrete objects (words, sentences, documents) into continuous
numeric vectors where geometric relationships encode semantic relationships.

Unlike a "sparse" vector, the
embedding vector is a fixed-size vector representation of the input's meaning in
high-dimensional space. Databases will often refer to it as a "dense vector" for
this reason.

## 2. Generate an embedding

### Structured document

Real social-media posts are messy: emojis, mixed case, punctuation, sometimes only an
image. Before we embed, we wrap each raw `Post` into a `PostDocument` (see
`src/data_models.py`). Think of it as a deconstructed post.

 The `PostDocument` can store  document attributes that are part of
 the unstructured text and turn them into structure documents. Examples: 

- `hashtag` - #NotIncludedInSampleData 
- `@mention` - Also not mentioned in our sample data 
- `emojies` - converted to text representation i.e. `happy_face_smiling_wink`
- `author_attributes` - For example, a bio.
- `publication_metadata` - The publisher, publishing date, copyright notice,
  citation reference, etc.
- `image_caption` — populated by a vision model, in our workshop, if the post has an image.
- `doc_embedding` — the 384-d vector that powers semantic search.

... any any other attribute you may want to include as a search attribute.

In [ ]:
from datetime import datetime, UTC

import emoji

from src.data_models import PostDocument

raw_text = "🐈" # Cat!

postdoc = PostDocument(
    post_id="demo-1",
    post_author="workshop",
    created_at=datetime.now(tzinfo=UTC).isoformat(),
    modified_at=datetime.now(tzinfo=UTC).isoformat(),
    post_text=raw_text,
)

print("Raw post_text:", postdoc.post_text)
clean_text = postdoc.preprocess_text()
print("Clean post_text:", clean_text)

### Sentence splitting

As we saw above, the embedding model treats text in much the same way whether it
is a word, a sentence, or a document. However, when comparing embeddings,
especially in search, it is important to consider the level of granularity to
use for your embeddings.

- Use sentence embeddings when your unit of retrieval is a short, focused claim.
- Use document embeddings when your unit of retrieval is the whole document and its
full context. 

In [ ]:
import numpy as np

# Typically you would use a more sophisticated sentence splitter.
# This is just a simple demonstration. See code in repo for examples of using
# SpaCy for sentence splitting.
def sentence_splitter(text: str) -> list[str]:
    """A very naive sentence splitter that just splits on periods."""
    return [s.strip() for s in text.split(".") if s.strip()]


raw_text1 = """
The first Supreme Cat Show took place in 1976.[3] Until then the GCCF itself did not organise cat shows, but licensed shows put on by the breed clubs and area clubs affiliated to it. The Supreme Cat Show was devised as a special show, only open to cats which had won an open class at another championship show under GCCF rules, much in the same way that Crufts is only open to winning dogs. The show grew in size each year until it became big enough to be held at the NEC, which has been its home ever since.
"""

postdoc1 = PostDocument(
    post_id="demo-1",
    post_author="workshop",
    created_at=datetime.now(tzinfo=UTC).isoformat(),
    modified_at=datetime.now(tzinfo=UTC).isoformat(),
    post_text=raw_text1,
)

raw_text2 = """
Markets fell sharply after the earnings report. Investors sold tech stocks
throughout the afternoon. Analysts warned about near-term volatility. Investors
regained confidence with the latest jobs report, and the markets rebounded by
the end of the week. Another example of the market trading aggressively
sideways in anticipation of an inversion of the yield curve, which is often seen as a recession signal.
 """
postdoc2 = PostDocument(
    post_id="demo-2",
    post_author="workshop",
    created_at=datetime.now(tzinfo=UTC).isoformat(),
    modified_at=datetime.now(tzinfo=UTC).isoformat(),
    post_text=raw_text2,
)

# 1) Sentence embeddings -> one sentence per embedding
doc1_sentences = sentence_splitter(postdoc1.post_text)
doc2_sentences = sentence_splitter(postdoc2.post_text)

sent_emb_doc1 = model.encode(
    doc1_sentences,
    batch_size=8, # Let's use batching to speed up encoding multiple sentences
    show_progress_bar=True,
    convert_to_numpy=True,
    )
sent_emb_doc2 = model.encode(
    doc2_sentences,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    )


print("doc1_sentences shape:", sent_emb_doc1.shape)
print("doc2_sentences shape:", sent_emb_doc2.shape)

Notice the shape of the array is n-sentences x 384; one vector for each
sentence.  

A search for "yield curve" or "GCCF" would yield that sentence or
chunk of the document which is encoded at a finer level of granularity than a
document as a whole. Larger documents are often broken down into sentences or
paragraphs to improve fine grain matching.

To get a representation of the whole document, we average the sentence
embeddings.  

This can be useful for medium sized documents on one topic. You can get a more balanced document summary, especially for long
texts where one long-pass encoding might overemphasize certain regions or hit
token limits.

**NOTE: For this workshop the posts are short. So, we'll just embed each one as
it's own document.**

### Cleaning and preprocessing text

Cleaning the text cleans words, symbols and other attributes into text that can
be tokenized and read by the model.

In [ ]:
### Demo Posts for Quick Experimenting ###
# We create a few tiny PostDocument examples manually so you can test
# preprocess_text() and embedding behavior before touching real workshop data.

from datetime import datetime, UTC

postdocs = [
    PostDocument(
        post_id="demo-1",
        post_author="workshop",
        created_at=datetime.now(tzinfo=UTC).isoformat(),
        modified_at=datetime.now(tzinfo=UTC).isoformat(),
        post_text="#HighSpeedRail https://www.californiarailmap.com/",
        image_caption="A photo of a high-speed train speeding through the California countryside, showcasing the potential of modern transportation infrastructure."
    ),
    PostDocument(
        post_id="demo-2",
        post_author="workshop",
        created_at=datetime.now(tzinfo=UTC).isoformat(),
        modified_at=datetime.now(tzinfo=UTC).isoformat(),
        post_text="Today's fog is more than just beautiful; it’s also serving as a reminder to stay prepared and alert during these unpredictable weather events. @KarlTheFog keeps us informed about what to expect, so everyone can enjoy this view."
    ),
    PostDocument(
        post_id="demo-3",
        post_author="workshop",
        created_at=datetime.now(tzinfo=UTC).isoformat(),
        modified_at=datetime.now(tzinfo=UTC).isoformat(),
        post_text="✨ Shout out to Texas! Come on y'all... Share your joy with a smile and laughter every single day! 🧡",
        image_caption="View of the Austin City Limits music hall in downtown Austin, Texas"
    ),
]

print("\nRaw post_texts:")
for post in postdocs:
    print(post.post_text)

print("\nClean post_texts:")
for post in postdocs:
    print(post.preprocess_text())

### Exercise ###
Add or remove preprocessing steps to PostDoc.preprocess_text() to clean any
additional issues

In [ ]:
# 2) Average sentence embeddings to get a single document embedding representing the whole document
doc1_embedding = np.mean(sent_emb_doc1, axis=0)
doc2_embedding = np.mean(sent_emb_doc2, axis=0)

print("doc1_embedding shape:", doc1_embedding.shape)
print("doc2_embedding shape:", doc2_embedding.shape)

### Generating embeddings



Embeddings are returned as numpy arrays in the same order as the input texts. To
help with evaluation, speed tasks like topic modeling, etc, we often map them
back to the original texts that were passed to the embedding model. 

In the case of this workshop we map them back to the structured `PostDoc`

In [ ]:
postdocs

In [ ]:
# 1. Build the text string the model will see for each post
# extract_embedding_text combines cleaned post_text and image_caption.

def extract_embedding_text(postdoc: PostDocument) -> str:
    # Extract the cleaned text elements to embed
    text = postdoc.preprocess_text()
    # Check if the post has an image caption (i.e. image converted to text)
    caption = (postdoc.image_caption or "").strip()
    if text and caption:
        return f"{text} {caption}"
    return text or caption

# 2. Batch-encode all texts at once. Batching is much faster than calling
#    the model once per post — the GPU/CPU stays warm between texts.
texts = [extract_embedding_text(postdoc) for postdoc in postdocs]
embeddings = model.encode(
    texts,
    batch_size=8,
    show_progress_bar=True,
    convert_to_numpy=True,
    )

# 3. Attach each embedding back onto its PostDocument.
# Important: The data must be JSON-serializable for Elasticsearch
# Convert numpy → list so the vector is
for postdoc, embedding in zip(postdocs, embeddings, strict=True):
    postdoc.doc_embedding = embedding.tolist()

In [ ]:
# Reference solution for src.preprocess.PreprocessingPipeline.generate_embeddings
# (lifted from solutions/preprocess.py — adapt to the method body in src/preprocess.py)

def generate_embeddings(self, postdocs):
    """Embed all postdocs using the embedding model."""
    # 1. Build the text string the model will see for each post.
    #    extract_embedding_text combines cleaned post_text and image_caption.
    texts = [extract_embedding_text(postdoc) for postdoc in postdocs]

    # 2. Batch-encode all texts at once. Batching is much faster than calling
    #    the model once per post — the GPU/CPU stays warm between texts.
    embeddings = self.embedding_model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        convert_to_numpy=True,
    )

    # 3. Attach each embedding back onto its PostDocument. Convert numpy → list
    #    so the vector is JSON-serializable for Elasticsearch and disk storage.
    for postdoc, embedding in zip(postdocs, embeddings, strict=True):
        postdoc.doc_embedding = embedding.tolist()

    return postdocs

# Quick sanity check on a single post — confirms the whole pipeline runs.
embedding = model.encode(extract_embedding_text(postdoc), convert_to_numpy=True)
print(f"Single-post embedding shape: {embedding.shape}")

# Exercise: 

Time: 10 minutes

Using what you have learned, replace the placeholder code in this function to
generate a document embedding for a single post. 

[generate_embeddings](../src/preprocess.py#L135)

Create a function that generates the embedding for a single post. The, pipeline
will call that function to generate embeddings for all posts in the dataset.

`src/preprocessing.py` generate_embeddings()



## Run the full preprocessing script

**IMPORTANT:** This step must be done before training the topic model. 

```bash
uv run python -m src.preprocess
```

Then run the file: 
`uv run src/preprocessing.py`

This will preprocess all the posts in sample_posts.json and load them into the Elasticsearch index. They will now be available for search. Try running a sample search from the previous exercise. 